In [1]:
import pandas as pd
import numpy as np
import gc
import json
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

DATA_PROC = Path("../data/processed")
MODELS_DIR = Path("../models")

# 检查所有需要的文件都在
required_files = [
    "train_with_all_features.parquet",
    "item_text_clusters.csv",
    "item_text_clusters_mpnet.csv",
]
for f in required_files:
    path = DATA_PROC / f
    assert path.exists(), f"❌ 缺文件: {f}"
    print(f"✅ {f}: {path.stat().st_size/1e6:.1f} MB")

✅ train_with_all_features.parquet: 1744.6 MB
✅ item_text_clusters.csv: 2.9 MB
✅ item_text_clusters_mpnet.csv: 2.9 MB


In [2]:
%%time
# 加载 v2 时存好的完整数据 (1.6 GB parquet, 比 csv 快 10x)
print("⏳ 加载 train_with_all_features.parquet...")
df_main = pd.read_parquet(DATA_PROC / "train_with_all_features.parquet")
print(f"   {len(df_main):,} 行, {len(df_main.columns)} 列")
print(f"   内存: {df_main.memory_usage(deep=True).sum()/1e9:.2f} GB")

# 加载两个 cluster 文件
text_clusters_minilm = pd.read_csv(
    DATA_PROC / "item_text_clusters.csv",
    dtype={'parent_asin': 'str', 'text_cluster_id': 'int16'},
)
text_clusters_mpnet = pd.read_csv(
    DATA_PROC / "item_text_clusters_mpnet.csv",
    dtype={'parent_asin': 'str', 'text_cluster_id_mpnet': 'int16'},
)

# LEFT JOIN 两个 cluster 进来 (一次性, 复用同样的 df_main)
df_main = df_main.merge(text_clusters_minilm, on='parent_asin', how='left')
df_main = df_main.merge(text_clusters_mpnet, on='parent_asin', how='left')

# 验证
print(f"\n拼接后 shape: {df_main.shape}")
print(f"列名: {list(df_main.columns)}")
print(f"\nNaN 统计:")
print(df_main[['text_cluster_id', 'text_cluster_id_mpnet']].isnull().sum())

# 释放小 df
del text_clusters_minilm, text_clusters_mpnet
gc.collect()

⏳ 加载 train_with_all_features.parquet...
   24,653,142 行, 18 列
   内存: 3.62 GB

拼接后 shape: (24653142, 20)
列名: ['user_id', 'parent_asin', 'label', 'user_interaction_count', 'user_avg_rating', 'user_last_timestamp', 'item_interaction_count', 'item_avg_rating', 'item_last_timestamp', 'price', 'price_missing', 'title_length', 'n_categories', 'sub_category_id', 'brand_id', 'user_avg_price', 'user_price_diff', 'pop_x_activity', 'text_cluster_id', 'text_cluster_id_mpnet']

NaN 统计:
text_cluster_id          0
text_cluster_id_mpnet    0
dtype: int64
CPU times: user 6.37 s, sys: 4.56 s, total: 10.9 s
Wall time: 9.86 s


0

In [3]:
%%time
# 完全复用 v2 的切分参数, 保证可比性
train_idx, val_idx = train_test_split(
    df_main.index,
    test_size=0.2,
    random_state=42,
    stratify=df_main['label'],
)

print(f"train: {len(train_idx):,} 行")
print(f"val:   {len(val_idx):,} 行")
print(f"val 正样本率: {df_main.loc[val_idx, 'label'].mean():.4f}")

train: 19,722,513 行
val:   4,930,629 行
val 正样本率: 0.1667
CPU times: user 4.43 s, sys: 374 ms, total: 4.8 s
Wall time: 4.91 s


In [5]:
def train_lightgbm_v3(df, feature_cols, categorical_features, train_idx, val_idx, label_col='label', model_name='v3'):
    """
    统一的 v3 训练函数, 保证 MiniLM 和 mpnet 版本完全一致只差特征
    """
    X_train = df.loc[train_idx, feature_cols]
    y_train = df.loc[train_idx, label_col]
    X_val = df.loc[val_idx, feature_cols]
    y_val = df.loc[val_idx, label_col]
    
    print(f"\n{'='*60}")
    print(f"训练 LightGBM {model_name}")
    print(f"{'='*60}")
    print(f"特征数: {len(feature_cols)}")
    print(f"Categorical: {categorical_features}")
    print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")
    
    train_data = lgb.Dataset(
        X_train, label=y_train,
        categorical_feature=categorical_features,
    )
    val_data = lgb.Dataset(
        X_val, label=y_val,
        reference=train_data,
        categorical_feature=categorical_features,
    )
    
    # 完全复用 v2 参数, 控制变量
    params = {
        'objective': 'binary',
        'metric': ['binary_logloss', 'auc'],
        'learning_rate': 0.05,
        'num_leaves': 31,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'num_threads': 8,
    }
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=500,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20),
            lgb.log_evaluation(period=50),
        ],
    )
    
    print(f"\n✅ {model_name} 完成")
    print(f"   Best iteration: {model.best_iteration}")
    print(f"   Val AUC:    {model.best_score['val']['auc']:.4f}")
    print(f"   Val LogLoss:{model.best_score['val']['binary_logloss']:.4f}")
    
    return model

In [6]:
%%time
# v3a: v2 的 15 个特征 + text_cluster_id (MiniLM)
FEATURE_COLS_V3_MINILM = [
    # v2 的 15 个 (不变)
    'user_interaction_count', 'user_avg_rating', 'user_last_timestamp',
    'item_interaction_count', 'item_avg_rating', 'item_last_timestamp',
    'price', 'price_missing', 'title_length', 'n_categories',
    'sub_category_id', 'brand_id',
    'user_avg_price', 'user_price_diff', 'pop_x_activity',
    # 新增 1 个
    'text_cluster_id',
]
CATEGORICAL_V3_MINILM = ['sub_category_id', 'brand_id', 'price_missing', 'text_cluster_id']

model_v3_minilm = train_lightgbm_v3(
    df=df_main,
    feature_cols=FEATURE_COLS_V3_MINILM,
    categorical_features=CATEGORICAL_V3_MINILM,
    train_idx=train_idx,
    val_idx=val_idx,
    model_name='v3-MiniLM',
)

# 保存模型
model_v3_minilm.save_model('../models/lightgbm_v3_minilm.txt')
print(f"\n✅ 模型保存: lightgbm_v3_minilm.txt")


训练 LightGBM v3-MiniLM
特征数: 16
Categorical: ['sub_category_id', 'brand_id', 'price_missing', 'text_cluster_id']
X_train: (19722513, 16), X_val: (4930629, 16)
Training until validation scores don't improve for 20 rounds
[50]	train's binary_logloss: 0.384901	train's auc: 0.775466	val's binary_logloss: 0.384979	val's auc: 0.775307
[100]	train's binary_logloss: 0.374235	train's auc: 0.78895	val's binary_logloss: 0.374398	val's auc: 0.788606
[150]	train's binary_logloss: 0.368248	train's auc: 0.797135	val's binary_logloss: 0.368506	val's auc: 0.796621
[200]	train's binary_logloss: 0.364202	train's auc: 0.80255	val's binary_logloss: 0.364524	val's auc: 0.801936
[250]	train's binary_logloss: 0.361411	train's auc: 0.806287	val's binary_logloss: 0.361807	val's auc: 0.80557
[300]	train's binary_logloss: 0.360005	train's auc: 0.80812	val's binary_logloss: 0.360464	val's auc: 0.807312
[350]	train's binary_logloss: 0.358763	train's auc: 0.809739	val's binary_logloss: 0.35929	val's auc: 0.808833
[40

In [7]:
%%time
# v3b: 把 MiniLM cluster 换成 mpnet cluster
FEATURE_COLS_V3_MPNET = [
    # v2 的 15 个 (不变)
    'user_interaction_count', 'user_avg_rating', 'user_last_timestamp',
    'item_interaction_count', 'item_avg_rating', 'item_last_timestamp',
    'price', 'price_missing', 'title_length', 'n_categories',
    'sub_category_id', 'brand_id',
    'user_avg_price', 'user_price_diff', 'pop_x_activity',
    # 换成 mpnet 版
    'text_cluster_id_mpnet',
]
CATEGORICAL_V3_MPNET = ['sub_category_id', 'brand_id', 'price_missing', 'text_cluster_id_mpnet']

model_v3_mpnet = train_lightgbm_v3(
    df=df_main,
    feature_cols=FEATURE_COLS_V3_MPNET,
    categorical_features=CATEGORICAL_V3_MPNET,
    train_idx=train_idx,
    val_idx=val_idx,
    model_name='v3-mpnet',
)

# 保存
model_v3_mpnet.save_model('../models/lightgbm_v3_mpnet.txt')
print(f"\n✅ 模型保存: lightgbm_v3_mpnet.txt")
print(f"\n📊 v3 双版本对比:")
print(f"   v3-MiniLM: AUC 0.8116")
print(f"   v3-mpnet:  AUC {model_v3_mpnet.best_score['val']['auc']:.4f}")


训练 LightGBM v3-mpnet
特征数: 16
Categorical: ['sub_category_id', 'brand_id', 'price_missing', 'text_cluster_id_mpnet']
X_train: (19722513, 16), X_val: (4930629, 16)
Training until validation scores don't improve for 20 rounds
[50]	train's binary_logloss: 0.385215	train's auc: 0.774673	val's binary_logloss: 0.385296	val's auc: 0.774527
[100]	train's binary_logloss: 0.374403	train's auc: 0.788521	val's binary_logloss: 0.374579	val's auc: 0.78816
[150]	train's binary_logloss: 0.368607	train's auc: 0.796581	val's binary_logloss: 0.368878	val's auc: 0.796048
[200]	train's binary_logloss: 0.36435	train's auc: 0.802444	val's binary_logloss: 0.364698	val's auc: 0.801785
[250]	train's binary_logloss: 0.361505	train's auc: 0.806216	val's binary_logloss: 0.361923	val's auc: 0.805453
[300]	train's binary_logloss: 0.359273	train's auc: 0.809046	val's binary_logloss: 0.359755	val's auc: 0.808199
[350]	train's binary_logloss: 0.35813	train's auc: 0.810517	val's binary_logloss: 0.358669	val's auc: 0.809

In [8]:
# 特征重要性对比 - 看 BERT cluster 在两个版本中的排名
imp_minilm = pd.DataFrame({
    'feature': FEATURE_COLS_V3_MINILM,
    'gain': model_v3_minilm.feature_importance(importance_type='gain'),
    'split': model_v3_minilm.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False).reset_index(drop=True)

imp_mpnet = pd.DataFrame({
    'feature': FEATURE_COLS_V3_MPNET,
    'gain': model_v3_mpnet.feature_importance(importance_type='gain'),
    'split': model_v3_mpnet.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False).reset_index(drop=True)

print("=" * 70)
print("v3-MiniLM 特征重要性")
print("=" * 70)
print(imp_minilm.to_string(index=False))

print("\n" + "=" * 70)
print("v3-mpnet 特征重要性")
print("=" * 70)
print(imp_mpnet.to_string(index=False))

# 找 text_cluster 的排名
minilm_rank = imp_minilm[imp_minilm['feature']=='text_cluster_id'].index[0] + 1
mpnet_rank = imp_mpnet[imp_mpnet['feature']=='text_cluster_id_mpnet'].index[0] + 1
minilm_gain = imp_minilm[imp_minilm['feature']=='text_cluster_id']['gain'].values[0]
mpnet_gain = imp_mpnet[imp_mpnet['feature']=='text_cluster_id_mpnet']['gain'].values[0]

print(f"\n📊 text_cluster 在两个模型中的位置:")
print(f"   v3-MiniLM: 排名 #{minilm_rank}/16, gain {minilm_gain:,.0f}")
print(f"   v3-mpnet:  排名 #{mpnet_rank}/16, gain {mpnet_gain:,.0f}")

v3-MiniLM 特征重要性
               feature         gain  split
item_interaction_count 1.131014e+07   1184
   item_last_timestamp 4.070025e+06   1565
   user_last_timestamp 2.958151e+06   1941
       user_price_diff 2.091802e+06    776
        user_avg_price 1.853025e+06   1227
                 price 1.748132e+06    685
        pop_x_activity 1.623823e+06    501
user_interaction_count 1.370960e+06   1295
              brand_id 1.316050e+06   3100
       text_cluster_id 9.294790e+05   1589
          title_length 7.933265e+05    424
       item_avg_rating 2.433055e+05    284
       sub_category_id 1.122027e+05    185
       user_avg_rating 6.112720e+04    195
         price_missing 4.512998e+04     44
          n_categories 5.709354e+02      5

v3-mpnet 特征重要性
               feature         gain  split
item_interaction_count 1.127717e+07   1139
   item_last_timestamp 4.027080e+06   1542
   user_last_timestamp 3.100337e+06   1981
       user_price_diff 2.099781e+06    784
                 price

In [9]:
%%time
# 一次预测两个模型
df_main['y_pred_v3_minilm'] = np.nan
df_main['y_pred_v3_mpnet'] = np.nan

X_val_minilm = df_main.loc[val_idx, FEATURE_COLS_V3_MINILM]
X_val_mpnet  = df_main.loc[val_idx, FEATURE_COLS_V3_MPNET]

df_main.loc[val_idx, 'y_pred_v3_minilm'] = model_v3_minilm.predict(
    X_val_minilm, num_iteration=model_v3_minilm.best_iteration
)
df_main.loc[val_idx, 'y_pred_v3_mpnet'] = model_v3_mpnet.predict(
    X_val_mpnet, num_iteration=model_v3_mpnet.best_iteration
)

# 准备分用户 Top-K 计算的两份 val 集
df_val_minilm = df_main.loc[val_idx, ['user_id', 'label', 'y_pred_v3_minilm']].copy()
df_val_minilm.rename(columns={'y_pred_v3_minilm': 'y_pred'}, inplace=True)

df_val_mpnet = df_main.loc[val_idx, ['user_id', 'label', 'y_pred_v3_mpnet']].copy()
df_val_mpnet.rename(columns={'y_pred_v3_mpnet': 'y_pred'}, inplace=True)

# Sanity check (整体 AUC 应该对得上)
auc_minilm = roc_auc_score(df_val_minilm['label'], df_val_minilm['y_pred'])
auc_mpnet = roc_auc_score(df_val_mpnet['label'], df_val_mpnet['y_pred'])
print(f"Sanity check AUC:")
print(f"  v3-MiniLM: {auc_minilm:.4f} (训练时: 0.8116)")
print(f"  v3-mpnet:  {auc_mpnet:.4f} (训练时: 0.8122)")

Sanity check AUC:
  v3-MiniLM: 0.8116 (训练时: 0.8116)
  v3-mpnet:  0.8122 (训练时: 0.8122)
CPU times: user 5min 5s, sys: 3.06 s, total: 5min 9s
Wall time: 49.9 s


In [10]:
%%time
import numpy as np

def compute_user_metrics(group, k_list=[5, 10, 20]):
    sorted_group = group.sort_values('y_pred', ascending=False)
    labels = sorted_group['label'].values
    n_pos = labels.sum()
    n_total = len(labels)
    metrics = {}
    for k in k_list:
        k_actual = min(k, n_total)
        top_k_labels = labels[:k_actual]
        n_hits = top_k_labels.sum()
        metrics[f'precision@{k}'] = n_hits / k_actual if k_actual > 0 else 0
        metrics[f'recall@{k}'] = n_hits / n_pos if n_pos > 0 else np.nan
        gains = (2 ** top_k_labels - 1)
        discounts = 1 / np.log2(np.arange(k_actual) + 2)
        dcg = (gains * discounts).sum()
        n_pos_in_topk = min(n_pos, k_actual)
        ideal_labels = np.zeros(k_actual)
        ideal_labels[:int(n_pos_in_topk)] = 1
        ideal_gains = (2 ** ideal_labels - 1)
        idcg = (ideal_gains * discounts).sum()
        metrics[f'ndcg@{k}'] = dcg / idcg if idcg > 0 else np.nan
    return pd.Series(metrics)

print("⏳ 计算 v3-MiniLM 分用户指标 (~2.5 min)...")
user_metrics_v3_minilm = df_val_minilm.groupby('user_id').apply(
    compute_user_metrics, k_list=[5, 10, 20], include_groups=False,
)
summary_v3_minilm = user_metrics_v3_minilm.mean()
print(f"✅ v3-MiniLM 完成")

print("\n⏳ 计算 v3-mpnet 分用户指标 (~2.5 min)...")
user_metrics_v3_mpnet = df_val_mpnet.groupby('user_id').apply(
    compute_user_metrics, k_list=[5, 10, 20], include_groups=False,
)
summary_v3_mpnet = user_metrics_v3_mpnet.mean()
print(f"✅ v3-mpnet 完成")

⏳ 计算 v3-MiniLM 分用户指标 (~2.5 min)...
✅ v3-MiniLM 完成

⏳ 计算 v3-mpnet 分用户指标 (~2.5 min)...
✅ v3-mpnet 完成
CPU times: user 3min 35s, sys: 3.57 s, total: 3min 39s
Wall time: 3min 40s


In [11]:
# v2 的指标 (从 Day 5 报告里读, 或硬编码)
v1_metrics = {
    'precision@5': 0.2053, 'precision@10': 0.1776, 'precision@20': 0.1694,
    'recall@5': 0.8800, 'recall@10': 0.9650, 'recall@20': 0.9906,
    'ndcg@5': 0.7333, 'ndcg@10': 0.7678, 'ndcg@20': 0.7777,
}
v2_metrics = {
    'precision@5': 0.2097, 'precision@10': 0.1787, 'precision@20': 0.1698,
    'recall@5': 0.8941, 'recall@10': 0.9686, 'recall@20': 0.9916,
    'ndcg@5': 0.7690, 'ndcg@10': 0.7990, 'ndcg@20': 0.8078,
}

print("=" * 95)
print("🏆 5 模型完整对比")
print("=" * 95)

# 全局指标
print(f"\n📊 全局指标:")
print(f"{'模型':<15}{'特征数':<8}{'Val AUC':<12}{'Val LogLoss':<14}{'训练耗时':<10}")
print("-" * 60)
print(f"{'v0 baseline':<15}{6:<8}{0.7645:<12}{0.3875:<14}{'2.7 min':<10}")
print(f"{'v1 tuned':<15}{6:<8}{0.7738:<12}{0.3810:<14}{'13.5 min':<10}")
print(f"{'v2 meta+cross':<15}{15:<8}{0.8100:<12}{0.3582:<14}{'14.1 min':<10}")
print(f"{'v3-MiniLM':<15}{16:<8}{0.8116:<12}{0.3571:<14}{'14.4 min':<10}")
print(f"{'v3-mpnet':<15}{16:<8}{0.8122:<12}{0.3567:<14}{'14.9 min':<10}")

# 分用户指标对比 (v1 → v2 → v3-MiniLM → v3-mpnet)
print(f"\n📊 分用户指标演进 (v1 → v2 → v3-MiniLM → v3-mpnet):")
print(f"{'指标':<15}{'v1':<10}{'v2':<10}{'v3-MiniLM':<12}{'v3-mpnet':<12}{'v3-mp vs v2':<13}")
print("-" * 72)

for metric in ['ndcg@5', 'ndcg@10', 'ndcg@20', 'precision@10', 'recall@10']:
    v1 = v1_metrics[metric]
    v2 = v2_metrics[metric]
    v3m = summary_v3_minilm[metric]
    v3p = summary_v3_mpnet[metric]
    delta = v3p - v2
    sign = '+' if delta >= 0 else ''
    print(f"{metric:<15}{v1:<10.4f}{v2:<10.4f}{v3m:<12.4f}{v3p:<12.4f}{sign}{delta:<.4f}")

# 总结
print(f"\n💡 核心发现:")
v3_minilm_auc = 0.8116
v3_mpnet_auc = 0.8122
v2_auc = 0.8100
print(f"   1. v3-mpnet 比 v3-MiniLM 高 {(v3_mpnet_auc - v3_minilm_auc):.4f} AUC")
print(f"   2. v3-mpnet 比 v2 高 {(v3_mpnet_auc - v2_auc):.4f} AUC (BERT cluster 边际收益)")
print(f"   3. 工程成本: mpnet 比 MiniLM 编码慢 2.7x, 模型大 5x")
print(f"   4. 综合 ROI: MiniLM 性价比胜出")

🏆 5 模型完整对比

📊 全局指标:
模型             特征数     Val AUC     Val LogLoss   训练耗时      
------------------------------------------------------------
v0 baseline    6       0.7645      0.3875        2.7 min   
v1 tuned       6       0.7738      0.381         13.5 min  
v2 meta+cross  15      0.81        0.3582        14.1 min  
v3-MiniLM      16      0.8116      0.3571        14.4 min  
v3-mpnet       16      0.8122      0.3567        14.9 min  

📊 分用户指标演进 (v1 → v2 → v3-MiniLM → v3-mpnet):
指标             v1        v2        v3-MiniLM   v3-mpnet    v3-mp vs v2  
------------------------------------------------------------------------
ndcg@5         0.7333    0.7690    0.7702      0.7707      +0.0017
ndcg@10        0.7678    0.7990    0.8001      0.8006      +0.0016
ndcg@20        0.7777    0.8078    0.8089      0.8093      +0.0015
precision@10   0.1776    0.1787    0.1788      0.1788      +0.0001
recall@10      0.9650    0.9686    0.9688      0.9690      +0.0004

💡 核心发现:
   1. v3-mpnet 比 v3-Mini

In [12]:
import json
from datetime import datetime

# 完整 Day 6 对比报告
day6_report = {
    'date': datetime.now().strftime('%Y-%m-%d'),
    'day': 'Day 6',
    'experiment_type': 'BERT text embedding comparison',
    
    'models': {
        'v0_baseline': {'features': 6, 'val_auc': 0.7645, 'val_logloss': 0.3875},
        'v1_tuned':    {'features': 6, 'val_auc': 0.7738, 'val_logloss': 0.3810},
        'v2_meta_cross': {'features': 15, 'val_auc': 0.8100, 'val_logloss': 0.3582},
        'v3_minilm': {
            'features': 16,
            'val_auc': float(auc_minilm),
            'val_logloss': float(model_v3_minilm.best_score['val']['binary_logloss']),
            'extra_feature': 'text_cluster_id (MiniLM-L6 384d -> KMeans 50)',
        },
        'v3_mpnet': {
            'features': 16,
            'val_auc': float(auc_mpnet),
            'val_logloss': float(model_v3_mpnet.best_score['val']['binary_logloss']),
            'extra_feature': 'text_cluster_id_mpnet (mpnet-base 768d -> KMeans 50)',
        },
    },
    
    'user_level_metrics_v3_minilm': {k: float(v) for k, v in summary_v3_minilm.items()},
    'user_level_metrics_v3_mpnet': {k: float(v) for k, v in summary_v3_mpnet.items()},
    
    'feature_importance_v3_mpnet_top5': imp_mpnet.head(5).to_dict(orient='records'),
    
    'key_findings': {
        'finding_1': f'mpnet 比 MiniLM AUC 高 {auc_mpnet - auc_minilm:.4f}, 但工程成本 5x (编码 2.7x, 模型 5x)',
        'finding_2': f'v3 比 v2 只涨 {auc_mpnet - 0.81:.4f}, 远低于 v2 比 v1 涨 0.036',
        'finding_3': '信号冗余假设: sub_category_id 已捕捉主要语义,BERT cluster 增量有限',
        'finding_4': '特征工程边际收益曲线: +0.036 (v2) → +0.002 (v3)',
        'production_decision': 'MiniLM 性价比胜出, 0.0006 AUC 不值 5x 工程成本',
    },
    
    'next_steps': [
        '考虑用户行为序列特征 (用户历史交互的 BERT 平均)',
        '尝试 200 个 cluster (更细粒度)',
        '加图片 CLIP embedding (Day 7)',
        '考虑深度模型 DIN/DeepFM',
    ],
}

with open('../models/day6_v3_comparison.json', 'w') as f:
    json.dump(day6_report, f, indent=2, ensure_ascii=False, default=str)

print("✅ Day 6 完整报告已保存: models/day6_v3_comparison.json")
print(f"\n🎯 Day 6 最终成果:")
print(f"   最佳模型: v3-mpnet (Val AUC {auc_mpnet:.4f})")
print(f"   推荐生产选择: v3-MiniLM (ROI 最优)")

✅ Day 6 完整报告已保存: models/day6_v3_comparison.json

🎯 Day 6 最终成果:
   最佳模型: v3-mpnet (Val AUC 0.8122)
   推荐生产选择: v3-MiniLM (ROI 最优)
